In [8]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
import psycopg2
import joblib


# Database configuration
db_config = {
    'dbname': 'postgres',
    'user': 'postgres',
    'password': 'DBP@ss',
    'host': '10.134.178.167',
    'port': '5432'
}

# Query to load relevant data
query = """
    SELECT delivery_finished, delivery_started, total_delivery_time, status, number_of_obstacles
    FROM deliveries
    WHERE status = 'COMPLETED'
"""

# Function to load data from database
def load_data_from_db():
    try:
        # Establish connection to PostgreSQL
        conn = psycopg2.connect(**db_config)

        # Fetch the data into a Pandas DataFrame
        with conn.cursor() as cursor:
            cursor.execute(query)
            colnames = [desc[0] for desc in cursor.description]  # Fetch column names
            rows = cursor.fetchall()
            df = pd.DataFrame(rows, columns=colnames)

        conn.close()

        # Process datetime fields
        df['delivery_finished'] = pd.to_datetime(df['delivery_finished'])
        df['hour'] = df['delivery_finished'].dt.hour
        df['day_of_week'] = df['delivery_finished'].dt.dayofweek
        df['month'] = df['delivery_finished'].dt.month

        # Filter for completed deliveries and process features
        df = df.dropna(subset=['total_delivery_time', 'number_of_obstacles'])
        X = df[['hour', 'day_of_week', 'month', 'number_of_obstacles']]
        y = df[['total_delivery_time']]

        return X, y
    except Exception as e:
        print(f"Error fetching data from database: {e}")
        exit(1)


# Load data
X, y = load_data_from_db()

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature preprocessing
feature_transformer = ColumnTransformer([
    ('num', StandardScaler(), ['hour', 'day_of_week', 'month'])
])

# Pipeline definition
pipeline = Pipeline(steps=[
    ('preprocessor', feature_transformer),
    ('model', MultiOutputRegressor(RandomForestRegressor(random_state=42)))
])

# Hyperparameter tuning
param_grid = {
    'model__estimator__n_estimators': [50, 100, 200],
    'model__estimator__max_depth': [None, 10, 20],
    'model__estimator__min_samples_split': [2, 5],
    'model__estimator__min_samples_leaf': [1, 2],
    'model__estimator__max_features': ['sqrt', 'log2']
}

# Perform grid search
grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Save the best model
best_model = grid_search.best_estimator_
joblib.dump(best_model, 'best_model.pkl')

print("Model trained and saved successfully.")

Model trained and saved successfully.
